In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path

In [2]:
METADATA_PATH = Path("../data/processed/graph_metadata.csv")
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")

metadata = pd.read_csv(METADATA_PATH)
cleaned_data = pd.read_csv(CLEANED_DATA_PATH)

print("Metadata shape:", metadata.shape)
print("Cleaned data shape:", cleaned_data.shape)

Metadata shape: (286095, 9)
Cleaned data shape: (286095, 68)


In [3]:
print("Metadata columns:")
print(metadata.columns.tolist())

print("\nCleaned data columns:")
print(cleaned_data.columns.tolist())

Metadata columns:
['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Label', 'Binary_Label']

Cleaned data columns:
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Fl

In [4]:
print("Binary label distribution:")
print(metadata["Binary_Label"].value_counts())

print("\nOriginal labels:")
print(metadata["Label"].value_counts())

Binary label distribution:
Binary_Label
1    158804
0    127291
Name: count, dtype: int64

Original labels:
Label
PortScan    158804
BENIGN      127291
Name: count, dtype: int64


In [5]:
print("Metadata rows:", len(metadata))
print("Cleaned data rows:", len(cleaned_data))

if len(metadata) != len(cleaned_data):
    raise ValueError(
        "Metadata and cleaned_data have different numbers of rows."
    )

print("Row counts match.")

Metadata rows: 286095
Cleaned data rows: 286095
Row counts match.


In [6]:
SOURCE_COL = "Source IP"
DESTINATION_COL = "Destination IP"

print("Source column:", SOURCE_COL)
print("Destination column:", DESTINATION_COL)

Source column: Source IP
Destination column: Destination IP


In [7]:
all_ips = pd.concat(
    [
        metadata[SOURCE_COL],
        metadata[DESTINATION_COL]
    ],
    ignore_index=True
).dropna().unique()

ip_to_node = {
    ip: idx
    for idx, ip in enumerate(all_ips)
}

print("Total unique IP nodes:", len(ip_to_node))

Total unique IP nodes: 4404


In [8]:
source_nodes = metadata[SOURCE_COL].map(ip_to_node)
destination_nodes = metadata[DESTINATION_COL].map(ip_to_node)

print("Missing source node IDs:", source_nodes.isna().sum())
print("Missing destination node IDs:", destination_nodes.isna().sum())

Missing source node IDs: 0
Missing destination node IDs: 0


In [9]:
valid_mask = (
    source_nodes.notna()
    & destination_nodes.notna()
    & (source_nodes != destination_nodes)
)

print("Valid flows:", valid_mask.sum())
print("Removed flows:", (~valid_mask).sum())

Valid flows: 286095
Removed flows: 0


In [10]:
metadata_valid = metadata.loc[valid_mask].reset_index(drop=True)
cleaned_data_valid = cleaned_data.loc[valid_mask].reset_index(drop=True)

source_nodes = source_nodes.loc[valid_mask].astype(int).reset_index(drop=True)
destination_nodes = destination_nodes.loc[valid_mask].astype(int).reset_index(drop=True)

print("Final flow count:", len(metadata_valid))

Final flow count: 286095


In [11]:
edge_index = torch.tensor(
    np.vstack(
        [
            source_nodes.to_numpy(),
            destination_nodes.to_numpy()
        ]
    ),
    dtype=torch.long
)

print("edge_index shape:", edge_index.shape)

edge_index shape: torch.Size([2, 286095])


In [12]:
edge_labels = torch.tensor(
    metadata_valid["Binary_Label"].to_numpy(),
    dtype=torch.long
)

print("Edge labels shape:", edge_labels.shape)

print("\nEdge label distribution:")
print(torch.bincount(edge_labels))

Edge labels shape: torch.Size([286095])

Edge label distribution:
tensor([127291, 158804])


In [13]:
graph_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label",
    "Binary_Label"
]

feature_columns = [
    col for col in cleaned_data_valid.columns
    if col not in graph_columns
]

print("Number of edge features:", len(feature_columns))

print("\nEdge feature columns:")
print(feature_columns)

Number of edge features: 67

Edge feature columns:
['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Si

In [14]:
edge_features_df = cleaned_data_valid[feature_columns].copy()

edge_features_df = edge_features_df.replace(
    [np.inf, -np.inf],
    np.nan
)

edge_features_df = edge_features_df.fillna(0)

edge_features = torch.tensor(
    edge_features_df.to_numpy(dtype=np.float32),
    dtype=torch.float32
)

print("Edge features shape:", edge_features.shape)

Edge features shape: torch.Size([286095, 67])


In [15]:
print("NaN values:", torch.isnan(edge_features).sum().item())
print("Infinite values:", torch.isinf(edge_features).sum().item())

NaN values: 0
Infinite values: 0


In [16]:
edge_graph = {
    "edge_index": edge_index,
    "edge_attr": edge_features,
    "edge_label": edge_labels
}

print("Edge graph created.")

print("\nedge_index:", edge_graph["edge_index"].shape)
print("edge_attr:", edge_graph["edge_attr"].shape)
print("edge_label:", edge_graph["edge_label"].shape)

Edge graph created.

edge_index: torch.Size([2, 286095])
edge_attr: torch.Size([286095, 67])
edge_label: torch.Size([286095])


In [17]:
node_ips = pd.DataFrame({
    "node_id": list(range(len(ip_to_node))),
    "ip_address": list(ip_to_node.keys())
})

print("Node table shape:", node_ips.shape)

node_ips.head()

Node table shape: (4404, 2)


,node_id,ip_address
0,0,192.168.10.12
1,1,192.168.10.16
2,2,192.168.10.50
3,3,192.168.10.14
4,4,192.168.10.17


In [18]:
edge_table = metadata_valid[
    [
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Source Port",
        "Destination Port",
        "Protocol",
        "Timestamp",
        "Label",
        "Binary_Label"
    ]
].copy()

edge_table["source_node"] = source_nodes
edge_table["destination_node"] = destination_nodes

print("Edge table shape:", edge_table.shape)

edge_table.head()

Edge table shape: (286095, 11)


,Flow ID,Source IP,Destination IP,Source Port,Destination Port,Protocol,Timestamp,Label,Binary_Label,source_node,destination_node
0,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,192.168.10.50,35396,22,6,7/7/2017 1:00,BENIGN,0,0,2
1,192.168.10.16-192.168.10.50-60058-22-6,192.168.10.16,192.168.10.50,60058,22,6,7/7/2017 1:00,BENIGN,0,1,2
2,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,192.168.10.50,35396,22,6,7/7/2017 1:00,BENIGN,0,0,2
3,192.168.10.12-192.168.10.50-35398-22-6,192.168.10.12,192.168.10.50,35398,22,6,7/7/2017 1:00,BENIGN,0,0,2
4,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.50,192.168.10.12,22,35396,6,7/7/2017 1:00,BENIGN,0,2,0


In [19]:
attack_edges = edge_table[
    edge_table["Binary_Label"] == 1
]

benign_edges = edge_table[
    edge_table["Binary_Label"] == 0
]

print("Benign edges:", len(benign_edges))
print("Attack edges:", len(attack_edges))

print("\nAttack source nodes:")
print(attack_edges["source_node"].nunique())

print("\nAttack destination nodes:")
print(attack_edges["destination_node"].nunique())

Benign edges: 127291
Attack edges: 158804

Attack source nodes:
1

Attack destination nodes:
1


In [20]:
OUTPUT_PATH = Path("../data/processed/edge_graph.pt")

torch.save(
    edge_graph,
    OUTPUT_PATH
)

print("Saved:", OUTPUT_PATH)

Saved: ..\data\processed\edge_graph.pt


In [21]:
EDGE_TABLE_PATH = Path("../data/processed/edge_table.csv")

edge_table.to_csv(
    EDGE_TABLE_PATH,
    index=False
)

print("Saved:", EDGE_TABLE_PATH)

Saved: ..\data\processed\edge_table.csv


In [22]:
NODE_TABLE_PATH = Path("../data/processed/edge_graph_nodes.csv")

node_ips.to_csv(
    NODE_TABLE_PATH,
    index=False
)

print("Saved:", NODE_TABLE_PATH)

Saved: ..\data\processed\edge_graph_nodes.csv


In [23]:
print("=" * 50)
print("CORRECTED EDGE-LEVEL GRAPH")
print("=" * 50)

print("Number of nodes:", len(node_ips))
print("Number of edges:", edge_index.shape[1])
print("Number of edge features:", edge_features.shape[1])

print("\nEdge labels:")
print("Benign:", (edge_labels == 0).sum().item())
print("Attack:", (edge_labels == 1).sum().item())

print("\nFiles created:")
print("1.", OUTPUT_PATH)
print("2.", EDGE_TABLE_PATH)
print("3.", NODE_TABLE_PATH)

CORRECTED EDGE-LEVEL GRAPH
Number of nodes: 4404
Number of edges: 286095
Number of edge features: 67

Edge labels:
Benign: 127291
Attack: 158804

Files created:
1. ..\data\processed\edge_graph.pt
2. ..\data\processed\edge_table.csv
3. ..\data\processed\edge_graph_nodes.csv
